# Lezione 8 — Whisper: dalla voce al testo

## Obiettivi di apprendimento
- Spiegare cos'è Whisper e perché è un sistema di riconoscimento vocale (ASR) robusto.
- Distinguere le dimensioni di modello di Whisper e il loro rapporto precisione/risorse.
- Usare Whisper tramite `transformers.pipeline` per trascrivere un audio reale.
- Riconoscere i casi d'uso reali di Whisper (sottotitolazione, trascrizione di riunioni, ecc.).

## Cosa costruiremo
Scaricheremo un audio di esempio pubblico e lo trascriveremo con Whisper
(`openai/whisper-tiny.en`) usando il pipeline di Hugging Face — la stessa via usata dal
progetto reale "AI Meeting Companion" di IBM — senza bisogno di alcuna chiave API.

## Whisper, in breve

Whisper è un sistema di riconoscimento automatico del parlato (ASR, *Automatic Speech
Recognition*) di OpenAI, addestrato con deep learning su una quantità enorme di audio reale.
Le sue tre caratteristiche chiave:
- **Alta precisione**, anche con audio di bassa qualità.
- **Supporto multilingue** automatico (non serve indicargli la lingua).
- **Robustezza rispetto al rumore** di fondo.

Esistono 5 dimensioni di modello: `tiny`, `base`, `small`, `medium`, `large` — a dimensione
maggiore corrisponde maggiore precisione ma più risorse necessarie. Casi d'uso reali:
sottotitolazione automatica di video, motori di ricerca vocali, trascrizione di riunioni,
supporto clienti multilingue.

Rivedi la spiegazione completa e l'analogia nel
[Capitolo 12 del manuale](../manual/12_whisper_voz_IT.md).

In [ ]:
# transformers e torch di solito sono preinstallati su Colab; li installiamo
# comunque per assicurarci versioni compatibili con il pipeline di ASR
!pip install -q transformers torch requests

In [ ]:
import requests

# Audio di esempio pubblico usato nel progetto reale "AI Meeting Companion" di IBM
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-GPXX04C6EN/Testing%20speech%20to%20text.mp3"

response = requests.get(url)
audio_file_path = "downloaded_audio.mp3"

if response.status_code == 200:
    with open(audio_file_path, "wb") as file:
        file.write(response.content)
    print("Audio scaricato correttamente.")
else:
    print("Impossibile scaricare l'audio. Codice:", response.status_code)

In [ ]:
from transformers import pipeline

# Come nel progetto reale AIMeetingCompanion_Project/simple_speech2text.py:
# usiamo Whisper tramite transformers.pipeline invece della libreria openai-whisper
# diretta, perché è più leggero da installare in un ambiente gratuito come Colab.
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
    chunk_length_s=30,
)

# batch_size=8 indica quanti frammenti di audio vengono elaborati alla volta
prediction = pipe(audio_file_path, batch_size=8)["text"]
print("Trascrizione:")
print(prediction)

## Riferimento: la libreria ufficiale `openai-whisper`

Il PDF sorgente di OpenAI Whisper documenta questo codice reale (anch'esso gratuito, ma più
pesante da installare su Colab):

```bash
pip install git+https://github.com/openai/whisper.git
```

```python
import whisper

model = whisper.load_model("base")
result = model.transcribe("audio_example.mp3")
print(result["text"])
```

## Riferimento: Whisper integrato in un server Flask

Riprendendo quanto visto nella lezione 7, ecco come si integrerebbe la trascrizione in un
endpoint Flask:

```python
from flask import Flask, request
import whisper

app = Flask(__name__)
model = whisper.load_model("base")

@app.route("/transcribe", methods=["POST"])
def transcribe_audio():
    audio_file = request.files["audio"]
    result = model.transcribe(audio_file)
    return {"transcription": result["text"]}

if __name__ == "__main__":
    app.run(debug=True)
```

## Dall'audio al riassunto: il caso d'uso completo

Il progetto reale "AI Meeting Companion" non si ferma alla trascrizione: passa il testo
trascritto a un LLM chiedendogli "i punti chiave della riunione". Nel materiale originale di
IBM, questo passaggio viene svolto da un modello a pagamento di watsonx. Nella pratica di
questa lezione risolverai lo stesso passaggio con un modello di riassunto gratuito di
Hugging Face.

## Riepilogo della lezione
- Whisper è un sistema ASR robusto, multilingue e preciso anche con audio di bassa qualità.
- Esistono 5 dimensioni di modello (`tiny` a `large`); usiamo `whisper-tiny.en` perché è il più leggero.
- `transformers.pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en")` è la via più pratica per usare Whisper su Colab, senza installare la libreria ufficiale.
- Il caso d'uso reale di questo blocco del corso è trascrivere + riassumere riunioni.

Continua con la [pratica di questa lezione](../notebooks_practica/clase08_practica_whisper_voz_IT.ipynb)
e rivedi il [Capitolo 12 del manuale](../manual/12_whisper_voz_IT.md).